# Module 14: Fast Legacy Hybrid - Performance & Visual Comparison

In this exercise, we demonstrate the architectural pattern of migrating legacy enterprise data (Esri REST APIs, GeoPackages) to cloud-native formats (GeoParquet) and high-performance compute engines (DuckDB).

We will explore:
1. **PyOgrio vs. Fiona**: Comparing bulk ingestion speeds for local files.
2. **The REST API Bottleneck**: Syncing live DC 311 Service Request data.
3. **Visual Benchmark**: Measuring the "time to map" for Live APIs vs. Cloud-Native Caches.

In [7]:
import os
import time
import leafmap
import geopandas as gpd
import pyogrio
import requests
import duckdb
import lonboard
from palettable.colorbrewer.diverging import RdYlGn_5

os.makedirs("data", exist_ok=True)
print("Setup complete.")

Setup complete.


## Round 1: Local Ingestion (The Bulk Pattern)
We compare the traditional `fiona` engine (row-based) against `pyogrio` (vectorized Arrow-backed).

In [8]:
gpkg_url = "https://github.com/opengeos/leafmap/raw/master/examples/data/countries.gpkg"
gpkg_path = "data/countries.gpkg"

if not os.path.exists(gpkg_path):
    leafmap.download_file(gpkg_url, gpkg_path)

print(f"Benchmarking: {gpkg_path}")

# Fiona
start = time.time()
df_fiona = gpd.read_file(gpkg_path, engine="fiona")
fiona_time = time.time() - start
print(f"Fiona Time: {fiona_time:.4f}s")

# PyOgrio
start = time.time()
df_pyogrio = gpd.read_file(gpkg_path, engine="pyogrio", use_arrow=True)
pyogrio_time = time.time() - start
print(f"PyOgrio Time: {pyogrio_time:.4f}s")

print(f"\n>>> RESULT: PyOgrio is {fiona_time / pyogrio_time:.1f}x faster.")

Benchmarking: data/countries.gpkg
Fiona Time: 0.0220s
PyOgrio Time: 0.0077s

>>> RESULT: PyOgrio is 2.9x faster.


## Round 2: Enterprise Sync (High Volume DC 311)
We pull 5,000 service requests from the DC GIS Feature Server and cache them locally.

In [9]:
service_url = "https://maps2.dcgis.dc.gov/dcgis/rest/services/DCGIS_DATA/ServiceRequests/FeatureServer/16"
parquet_path = "data/dc_311_cache.parquet"

params = {
    "where": "1=1",
    "outFields": "SERVICEREQUESTID,STREETADDRESS,WARD,SERVICECODEDESCRIPTION,PRIORITY",
    "f": "geojson",
    "resultRecordCount": "5000"
}

print(f"Syncing {params['resultRecordCount']} features from DC GIS...")
start = time.time()
response = requests.get(f"{service_url}/query", params=params)
df_sync = gpd.read_file(response.text, engine="pyogrio")
df_sync.to_parquet(parquet_path)
print(f"Sync Complete in {time.time() - start:.2f}s. Saved to {parquet_path}.")

Syncing 5000 features from DC GIS...
Sync Complete in 26.89s. Saved to data/dc_311_cache.parquet.


## Round 3: Visual Benchmark (Live API vs. Parquet Cache)
In this round, we simulate a user interactive action (e.g., clicking a filter) and measure the total time to render the results on the map.

### 3.1 Live Service Visualization (The "Legacy" Experience)
Every filter change requires a network roundtrip and a slow JSON parse.

In [10]:
target_code = "Bulk Collection"
print(f"ACTION: Filter map for '{target_code}' via LIVE API...")

live_params = params.copy()
live_params["where"] = f"SERVICECODEDESCRIPTION = '{target_code}'"

start_live = time.time()
res_live = requests.get(f"{service_url}/query", params=live_params)
df_live = gpd.read_file(res_live.text, engine="pyogrio")

m_live = lonboard.viz(df_live)
live_total_time = time.time() - start_live

print(f"\n>>> TOTAL TIME TO SIGHT (LIVE): {live_total_time:.4f}s")
m_live

ACTION: Filter map for 'Bulk Collection' via LIVE API...

>>> TOTAL TIME TO SIGHT (LIVE): 10.1980s


### 3.2 Local Cache Visualization (The "Fast Hybrid" Experience)
Data is scanned via DuckDB from local GeoParquet. No network latency, no JSON parsing overhead.

In [11]:
print(f"ACTION: Filter map for '{target_code}' via LOCAL CACHE...")

con = duckdb.connect()
start_local = time.time()

query = f"SELECT * FROM read_parquet('{parquet_path}') WHERE SERVICECODEDESCRIPTION = '{target_code}'"
df_local_raw = con.execute(query).df()

# Re-link geometry (in a real app, geometry is stored in the parquet)
# Lonboard viz uses the geometry column automatically
# Since we saved to parquet via geopandas, the geometry is preserved.
gdf_local = gpd.read_parquet(parquet_path)
gdf_filtered = gdf_local[gdf_local['SERVICECODEDESCRIPTION'] == target_code]

m_local = lonboard.viz(gdf_filtered)
local_total_time = time.time() - start_local

print(f"\n>>> TOTAL TIME TO SIGHT (LOCAL): {local_total_time:.4f}s")
m_local

ACTION: Filter map for 'Bulk Collection' via LOCAL CACHE...

>>> TOTAL TIME TO SIGHT (LOCAL): 0.0413s


## Conclusion

Compare the two maps above. Notice the substantial difference in "Time to Sight". 

By adopting the **Fast Legacy Hybrid** pattern, you unlock:
- **Instant Feedback Loops**: Change a filter, see the data immediately.
- **Reduced Infrastructure Costs**: No redundant hits to the expensive enterprise database.
- **Premium Experience**: The user never sees a loading spinner.